In [ ]:
using Gen
using Statistics
using Distributions

# R-hat computation function remains the same
function compute_rhat(chains)
    m = length(chains)
    n = length(chains[1])
    chain_means = [mean(chain) for chain in chains]
    grand_mean = mean(chain_means)
    W = mean([var(chain, corrected=true) for chain in chains])
    B = (n / (m - 1)) * sum((chain_mean - grand_mean)^2 for chain_mean in chain_means)
    var_hat = ((n - 1)/n)*W + (B/n)
    r_hat = sqrt(var_hat/W)
    return r_hat
end

# Model definition remains unchanged
@gen function eight_school_model(sigma)
    mu ~ normal(0, 1)
    tau ~ normal(2, 5)  # Half-Cauchy would need different implementation
    list_of_Eta = [{(:eta, i)} ~ normal(0, 1) for i=1:length(sigma)]
    for i in 1:length(sigma)
        theta = mu + tau * list_of_Eta[i]
        {(:y, i)} ~ normal(theta, sigma[i])
    end
end



function multi_variable_metropolis(trace, model, sigma, observations, eps_mu, eps_tau, eps_eta)
    # Extract current values of parameters
    mu_current = get_choices(trace)[:mu]
    tau_current = get_choices(trace)[:tau]
    eta_current = [get_choices(trace)[(:eta, i)] for i in 1:length(sigma)]
    
    
    # Propose new values for mu, tau, and eta
    mu_proposed = mu_current + eps_mu * randn()
    tau_proposed = tau_current +  eps_tau *  randn()
    eta_proposed = [eta_current[i] + eps_eta* randn() for i in 1:length(sigma)]
    
     # Create a temporary choice map with the proposed values
    temp_cm = choicemap(
        (:mu => mu_proposed),
        (:tau => tau_proposed)
    )
   
    # Add proposed eta values to the choice map
    for i in 1:length(sigma)
        
        temp_cm[(:eta, i)] = eta_proposed[i]
        # print("hoi") 
    end

     # Compute acceptance
    proposed_trace, = Gen.update(trace, (sigma,), (), temp_cm)
    acceptance_ratio = exp(get_score(proposed_trace) - get_score(trace))
    
    rand() < acceptance_ratio ? (proposed_trace, 1) : (trace, 0)
end


# Importance Sampling inference (fixed)
function do_inference_is(model, args, y_obs, num_samples)
      observations = choicemap()
    for (i, y) in enumerate(y_obs)
        observations[(:y, i)] = y
    end
    
    (traces, log_weights, _) = Gen.importance_sampling(
        model,
        args,
        observations,
        num_samples
    )
    
    # Gen already returns NORMALIZED log weights - no need for logsumexp
    weights = exp.(log_weights)
    
    # Direct weighted statistics (no resampling)
    mu_samples = [get_choices(t)[:mu] for t in traces]
    tau_samples = [get_choices(t)[:tau] for t in traces]
    
    return (mu_samples, tau_samples, weights)  # Return weights for proper analysis
end



# Metropolis-Hastings inference
function do_inference_mh(model, sigma, y_obs, num_iters)
    observations = choicemap()
    for (i, y) in enumerate(y_obs)
        observations[(:y, i)] = y
    end

    (trace, _) = generate(model, (sigma,), observations)
    accepted = 0
    mu_samples = []
    tau_samples = []

    # Select all parameters to update
    selection = select(:mu, :tau, (:eta, i for i in 1:length(sigma))...)

    for _ in 1:num_iters
        (new_trace, acc) = metropolis_hastings(trace, selection)
        accepted += acc
        trace = new_trace
        push!(mu_samples, trace[:mu])
        push!(tau_samples, trace[:tau])
    end

    return (mu_samples, tau_samples, accepted/num_iters)
end

# HMC inference
function do_inference_hmc(model, sigma, y_obs, num_iters; L=10, eps=0.1)
    observations = choicemap()
    for (i, y) in enumerate(y_obs)
        observations[(:y, i)] = y
    end

    (trace, _) = generate(model, (sigma,), observations)
    accepted = 0
    mu_samples = []
    tau_samples = []

    # Select all continuous parameters
    selection = select(:mu, :tau, (:eta, i for i in 1:length(sigma))...)

    for _ in 1:num_iters
        (new_trace, acc) = hmc(trace, selection; L=L, eps=eps)
        accepted += acc
        trace = new_trace
        push!(mu_samples, trace[:mu])
        push!(tau_samples, trace[:tau])
    end

    return (mu_samples, tau_samples, accepted/num_iters)
end


function do_inference(model, sigma, y_obs, num_iters,eps_mu, eps_tau, eps_eta)
    observations = choicemap()
    for (i, y) in enumerate(y_obs)
        observations[(:y, i)] = y
    end

    (trace, _) = generate(model, (sigma,), observations)
    accepted = 0
    mu_samples = []
    tau_samples = []

    # Store sampled values at each iteration
    for _ in 1:num_iters
        (trace, accepted_this_iter) = multi_variable_metropolis(trace, model, sigma, observations, eps_mu, eps_tau, eps_eta)
        accepted += accepted_this_iter
        
        # Store samples
        final_choices = get_choices(trace)
        push!(mu_samples, final_choices[:mu])
        push!(tau_samples, final_choices[:tau])
    end

    acceptance_rate = accepted / num_iters  # Compute acceptance rate

    return (mu_samples, tau_samples, acceptance_rate)
end

# Tuned version of your inference function
function do_inference_tuned(model, sigma, y_obs, num_iters)
    # Best parameters from analysis
     eps_mu=1 ;
    eps_tau= 1; 
    eps_eta= 1;
    
    return do_inference(model, sigma, y_obs, num_iters, 
                       eps_mu, eps_tau, eps_eta)Ill
end



function do_inference_adaptive(model, sigma, y_obs, num_iters; target_accept=0.3)
    observations = choicemap()
    for (i, y) in enumerate(y_obs)
        observations[(:y, i)] = y
    end

    (trace, _) = generate(model, (sigma,), observations)
    accepted = 0
    
    # Initialize ε with tuned values
    eps_mu, eps_tau, eps_eta = 1, 1, 1;
    mu_samples = []
    tau_samples = []
    
    # Adaptation settings
    adapt_window = 100
    adapt_factor = 1.05  # Conservative adjustment
    
    for iter in 1:num_iters
        (trace, acc) = multi_variable_metropolis(trace, model, sigma, observations, eps_mu, eps_tau, eps_eta)
        accepted += acc
        
        # Adapt ε during first 20% of iterations
        if iter <= 0.2*num_iters && iter % adapt_window == 0
            current_accept = accepted / adapt_window
            if current_accept < target_accept
                eps_mu *= 0.95; eps_tau *= 0.95; eps_eta *= 0.95  # Smaller steps
            else
                eps_mu *= adapt_factor; eps_tau *= adapt_factor; eps_eta *= adapt_factor
            end
            accepted = 0
        end
        
        # Store samples post-burn-in
        iter > 0.2*num_iters && (push!(mu_samples, trace[:mu]); push!(tau_samples, trace[:tau]))
    end
    
    return (mu_samples, tau_samples, accepted / num_iters)
end;


#

In [ ]:


sigma = [15, 10, 16, 11, 9, 11, 10, 18]
y_obs = [28, 8, -3, 7, -1, 1, 18, 12]
num_chains = 20
num_samples = 2000

# Add custom Metropolis to methods list
methods = [
    ("Importance Sampling", :is),
    ("Gen Metropolis", :mh),
    ("Custom Metropolis", :custom_mh),
    ("Hamiltonian MC", :hmc)
]

# Modified main comparison loop
for (method_name, method) in methods
    println("\n=== $method_name ===")
    
    chains_mu = []
    chains_tau = []
    acc_rates = []
    
    for _ in 1:num_chains
        if method == :is
            mu, tau, _ = do_inference_is(eight_school_model, (sigma,), y_obs, num_samples)
            push!(chains_mu, mu)
            push!(chains_tau, tau)
        elseif method == :mh
            mu, tau, acc = do_inference_mh(eight_school_model, sigma, y_obs, num_samples)
            push!(chains_mu, mu)
            push!(chains_tau, tau)
            push!(acc_rates, acc)
        elseif method == :custom_mh
            # Use optimal epsilon from previous testing
            
            mu, tau, acc = do_inference_adaptive(eight_school_model, sigma, y_obs, Int(ceil(num_samples / 0.8)))
            push!(chains_mu, mu)
            push!(chains_tau, tau)
            push!(acc_rates, acc)
        else
            mu, tau, acc = do_inference_hmc(eight_school_model, sigma, y_obs, num_samples)
            push!(chains_mu, mu)
            push!(chains_tau, tau)
            push!(acc_rates, acc)
        end
    end

    # Diagnostics
    if method != :is
        println("μ - R̂: ", compute_rhat(chains_mu))
        println("τ - R̂: ", compute_rhat(chains_tau))
    end
    println("μ mean: ", mean(vcat(chains_mu...)))
        
    println("τ mean: ", mean(vcat(chains_tau...)))
    
    if method != :is
        println("Acceptance rate: ", mean(acc_rates))
    end
end

# Additional analysis for custom Metropolis with different eps values
println("\nCustom Metropolis Epsilon Analysis:")
for eps in [0.1, 0.5, 1.0, 1.5, 2.0]
    chains_mu = []
    chains_tau = []
    acc_rates = []
    
    for _ in 1:num_chains
        mu, tau, acc = do_inference(eight_school_model, sigma, y_obs, num_samples, eps, eps, eps)
        push!(chains_mu, mu)
        push!(chains_tau, tau)
        push!(acc_rates, acc)
    end
    
    println("\nε = ", eps)
    println("μ R̂: ", compute_rhat(chains_mu))
    println("τ R̂: ", compute_rhat(chains_tau))
    println("Acceptance rate: ", mean(acc_rates))
end